# DynaRapid Pixel Filter for MIPI Camera Video Pipeline
---

## Aim

This notebook illustrates that a DynaRapid-generated filter can be inserted into a video pipeline running at 300 MHz. This notebook is based off of Xilinx' notebook found [here](https://github.com/Xilinx/AUP-ZU3/blob/main/base/notebooks/video/mipi_to_displayport.ipynb).

---

## Filter List

In [ ]:
# Lists the filters available
import os

all_filters_path = os.path.join("/home", "xilinx", "pynq", "overlays")
all_filters = [d for d in os.listdir(all_filters_path) if os.path.isdir(os.path.join(all_filters_path, d)) and d != "__pycache__"]
print('\n'.join(sorted(all_filters)))

## Filter Selection

In [ ]:
filter = "passthrough"

## Load _base_ Overlay, Import Video and Image Libraries

In [ ]:
import os

filter_path = os.path.join(all_filters_path, filter, "base.bit")

# Make sure filter exists
assert os.path.isfile(filter_path), f"[ERROR] Filter bitstream couldn't be found at '{filter_path}'"

import importlib
BaseOverlay = getattr(importlib.import_module(f"pynq.overlays.{filter}"), "BaseOverlay")

from pynq.devicetree import DeviceTreeSegment
from pynq.lib.video import *
import PIL.Image

# Reset PL - takes ~2 seconds, hence why it's commented out, but you can reenable if you so desire
from pynq import PL
PL.reset()

base = BaseOverlay(filter_path)
print(f"[INFO] Finished loading bitstream from '{filter}'")

## Set up the MIPI Camera and DynaRapid Filter

In [ ]:
# Grab a handler to the MIPI hierarchy, this will initialize the camera
mipi = base.mipi

# Setup the camera for 1280x720 mode with 24-bit pixels
videomode = VideoMode(1280, 720, 24)
mipi.configure(videomode)

dyna_accel = mipi.dyna_accel_1

# Configure accelerator
dyna_accel.write(0x10, videomode.height)
dyna_accel.write(0x18, videomode.width)

height = dyna_accel.read(0x10)
width = dyna_accel.read(0x18)

# Make sure the accelerators were configured correctly
assert height == videomode.height and width == videomode.width, f"[ERROR] Filter wasn't configured properly ({width}, {height}) - expected ({videomode.width}, {videomode.height})"

# Start reading from the camera
mipi.start()

print(f"Configured resolution: {width} x {height}")

## Capture Frame and Display in the Notebook


In [ ]:
frame = mipi.readframe()

# Note that the channels are arranged differently that what PIL expects, for that reason we reorder them 
PIL.Image.fromarray(frame[:,:,[2,1,0]])

## Enable and Start the DisplayPort Output

Since it takes about 2 seconds for the hardware to handshake with the connected DisplayPort display after turning it on wait for that long

In [ ]:
displayport = DisplayPort()
displayport.configure(videomode, PIXEL_RGB)

Read and display 3000 frames

In [ ]:
import time
num_frames = 3000
start = time.time()

for _ in range (num_frames):
    frame = displayport.newframe()
    frame[:] = mipi.readframe()
    displayport.writeframe(frame)

end = time.time()
duration = end - start
print(f"Took {duration} seconds at {num_frames / duration} FPS")

## Clean up DisplayPort Output and MIPI Camera Buffer

In [ ]:
displayport.close()
mipi.stop()

Copyright (c) 2022 Xilinx, Inc 
<br>
Copyright (C) 2022-2025 Advanced Micro Devices, Inc. 
<br>
SPDX-License-Identifier: BSD-3-Clause 

---

---